# 02. Build variant-level feature matrix

This notebook starts from the already prepared `mtDNA_master.csv` and creates several variant-level matrices for clustering.

Main idea:

- one row = one mtDNA variant
- `variant_id` = variant identifier
- modeling columns = numeric features only
- labels such as `validation_label`, `is_pathogenic_dataset9`, `is_neutral_dataset8`, and `is_artifact_prone_site` are not used in the primary unsupervised matrix

## 1. Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

MASTER_PATH = Path("results/01_master/mtDNA_master.csv")
OUT_DIR = Path("results/02_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MTDNA_LENGTH = 16569
AF_EPSILON = 1e-8

## 2. Load master file

In [ ]:
master = pd.read_csv(MASTER_PATH, low_memory=False)

print(master.shape)
master.head()

In [ ]:
master.columns

## 3. Minimal QC of required columns

In [ ]:
required_cols = [
    "position", "reference", "alternate", "consequence",
    "mlc_score", "variant_id",
    "gnomad_observed", "gnomad_filter", "AN",
    "gnomad_homoplasmic_ac", "gnomad_heteroplasmic_ac",
    "gnomad_homoplasmic_af", "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple", "gnomad_max_heteroplasmy",
    "helix_counts_hom", "helix_af_hom",
    "helix_counts_het", "helix_af_het",
    "helix_mean_arf", "helix_max_arf", "helix_observed",
    "gene", "feature", "vep",
    "is_disease_suspected_dataset3", "is_neutral_dataset8",
    "is_pathogenic_dataset9", "validation_label",
    "in_regional_constraint", "in_noncoding_constraint",
    "is_artifact_prone_site"
]

missing = [c for c in required_cols if c not in master.columns]
missing

In [ ]:
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

## 4. Standardize core columns

This part keeps the notebook explicit. We only use small helper functions for repeated cleaning operations.

In [ ]:
def to_num(s, default=np.nan):
    return pd.to_numeric(s, errors="coerce").fillna(default)

def to_bool01(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) > 0).astype(int)
    x = s.astype("string").fillna("").str.strip().str.lower()
    return x.isin({"1", "true", "t", "yes", "y", "observed", "pass"}).astype(int)

def log10_af(s, eps=AF_EPSILON):
    return np.log10(pd.to_numeric(s, errors="coerce").fillna(0).clip(lower=0) + eps)

def log10_count(s):
    return np.log10(pd.to_numeric(s, errors="coerce").fillna(0).clip(lower=0) + 1)

In [ ]:
master = master.copy()

master["position"] = pd.to_numeric(master["position"], errors="coerce").astype("Int64")
master["reference"] = master["reference"].astype("string").fillna("").str.strip().str.upper()
master["alternate"] = master["alternate"].astype("string").fillna("").str.strip().str.upper()
master["consequence"] = master["consequence"].astype("string").fillna("").str.strip()
master["variant_id"] = master["variant_id"].astype("string").fillna("").str.strip()

empty_id = master["variant_id"].eq("")
master.loc[empty_id, "variant_id"] = (
    "m."
    + master.loc[empty_id, "position"].astype(str)
    + master.loc[empty_id, "reference"].astype(str)
    + ">"
    + master.loc[empty_id, "alternate"].astype(str)
)

invalid_core = (
    master["position"].isna()
    | master["reference"].eq("")
    | master["alternate"].eq("")
    | master["variant_id"].eq("")
)

print("Invalid core rows:", int(invalid_core.sum()))
master = master.loc[~invalid_core].copy()

print(master.shape)
master[["position", "reference", "alternate", "variant_id", "consequence"]].head()

## 5. Check and collapse duplicated variants

The master file should ideally already be one row per variant. If duplicated `variant_id` values remain, we keep the first non-null value per column.

In [ ]:
dup_n = int(master["variant_id"].duplicated().sum())
dup_n

In [ ]:
if dup_n > 0:
    master = (
        master
        .groupby("variant_id", as_index=False, sort=False)
        .agg(lambda x: x.dropna().iloc[0] if x.dropna().shape[0] else np.nan)
    )

print(master.shape)
print("Duplicated variant_id:", int(master["variant_id"].duplicated().sum()))

## 6. Create full feature table

In [ ]:
features = pd.DataFrame(index=master.index)

features["variant_id"] = master["variant_id"].astype(str)
features["position"] = master["position"].astype(int)
features["reference"] = master["reference"].astype(str)
features["alternate"] = master["alternate"].astype(str)
features["consequence"] = master["consequence"].astype(str)
features["gene"] = master["gene"].astype("string").fillna("").str.strip()
features["feature"] = master["feature"].astype("string").fillna("").str.strip()

features.head()

## 7. Variant type features

In [ ]:
ref = features["reference"].astype("string")
alt = features["alternate"].astype("string")

features["ref_len"] = ref.str.len().fillna(0).astype(int)
features["alt_len"] = alt.str.len().fillna(0).astype(int)

features["is_snv"] = ((features["ref_len"] == 1) & (features["alt_len"] == 1)).astype(int)
features["is_indel"] = (
    (features["ref_len"] != features["alt_len"])
    | (features["ref_len"] > 1)
    | (features["alt_len"] > 1)
).astype(int)

substitution = ref + ">" + alt
transitions = {"A>G", "G>A", "C>T", "T>C"}
valid_bases = ref.isin(["A", "C", "G", "T"]) & alt.isin(["A", "C", "G", "T"])

features["is_transition"] = (features["is_snv"].eq(1) & substitution.isin(transitions)).astype(int)
features["is_transversion"] = (
    features["is_snv"].eq(1) & valid_bases & ~substitution.isin(transitions)
).astype(int)

features[["variant_id", "is_snv", "is_indel", "is_transition", "is_transversion"]].head()

## 8. Circular mtDNA position features

Raw `position` can dominate clustering. Circular encoding is safer because mtDNA is circular.

In [ ]:
angle = 2 * np.pi * (features["position"] / MTDNA_LENGTH)

features["mt_pos_sin"] = np.sin(angle)
features["mt_pos_cos"] = np.cos(angle)

features[["position", "mt_pos_sin", "mt_pos_cos"]].head()

## 9. MLC / local constraint features

In [ ]:
features["mlc_score"] = to_num(master["mlc_score"])
features["mlc_position_score"] = to_num(master["mlc_position_score"])

features["mlc_score_missing"] = features["mlc_score"].isna().astype(int)
features["mlc_position_score_missing"] = features["mlc_position_score"].isna().astype(int)

features["mlc_score"] = features["mlc_score"].fillna(features["mlc_score"].median())
features["mlc_position_score"] = features["mlc_position_score"].fillna(features["mlc_position_score"].median())

features[["mlc_score", "mlc_position_score", "mlc_score_missing", "mlc_position_score_missing"]].describe().T

## 10. gnomAD features

In [ ]:
features["gnomad_observed"] = to_bool01(master["gnomad_observed"])

gnomad_filter = master["gnomad_filter"].astype("string").fillna("").str.strip()
features["gnomad_filter_pass"] = gnomad_filter.str.upper().eq("PASS").astype(int)
features["gnomad_filter_missing"] = gnomad_filter.eq("").astype(int)

features["gnomad_AN"] = to_num(master["AN"], default=0).clip(lower=0)
features["gnomad_homoplasmic_ac"] = to_num(master["gnomad_homoplasmic_ac"], default=0).clip(lower=0)
features["gnomad_heteroplasmic_ac"] = to_num(master["gnomad_heteroplasmic_ac"], default=0).clip(lower=0)
features["gnomad_total_ac"] = features["gnomad_homoplasmic_ac"] + features["gnomad_heteroplasmic_ac"]

hom_af_from_counts = features["gnomad_homoplasmic_ac"] / features["gnomad_AN"].replace(0, np.nan)
het_af_from_counts = features["gnomad_heteroplasmic_ac"] / features["gnomad_AN"].replace(0, np.nan)
combined_af_from_counts = features["gnomad_total_ac"] / features["gnomad_AN"].replace(0, np.nan)

features["gnomad_homoplasmic_af"] = to_num(master["gnomad_homoplasmic_af"]).fillna(hom_af_from_counts).fillna(0).clip(lower=0)
features["gnomad_heteroplasmic_af"] = to_num(master["gnomad_heteroplasmic_af"]).fillna(het_af_from_counts).fillna(0).clip(lower=0)
features["gnomad_combined_af"] = to_num(master["gnomad_combined_af_simple"]).fillna(combined_af_from_counts).fillna(0).clip(lower=0)
features["gnomad_max_heteroplasmy"] = to_num(master["gnomad_max_heteroplasmy"], default=0).clip(lower=0)

features["log_gnomad_homoplasmic_af"] = log10_af(features["gnomad_homoplasmic_af"])
features["log_gnomad_heteroplasmic_af"] = log10_af(features["gnomad_heteroplasmic_af"])
features["log_gnomad_combined_af"] = log10_af(features["gnomad_combined_af"])
features["log_gnomad_total_ac"] = log10_count(features["gnomad_total_ac"])
features["log_gnomad_AN"] = log10_count(features["gnomad_AN"])

features["gnomad_has_homoplasmic_carrier"] = (features["gnomad_homoplasmic_ac"] > 0).astype(int)
features["gnomad_has_heteroplasmic_carrier"] = (features["gnomad_heteroplasmic_ac"] > 0).astype(int)

features[[
    "gnomad_observed", "gnomad_filter_pass", "gnomad_total_ac",
    "gnomad_combined_af", "gnomad_max_heteroplasmy"
]].describe().T

## 11. Helix features

In [ ]:
features["helix_observed"] = to_bool01(master["helix_observed"])

features["helix_counts_hom"] = to_num(master["helix_counts_hom"], default=0).clip(lower=0)
features["helix_counts_het"] = to_num(master["helix_counts_het"], default=0).clip(lower=0)
features["helix_total_counts"] = features["helix_counts_hom"] + features["helix_counts_het"]

features["helix_af_hom"] = to_num(master["helix_af_hom"], default=0).clip(lower=0)
features["helix_af_het"] = to_num(master["helix_af_het"], default=0).clip(lower=0)
features["helix_combined_af"] = pd.concat(
    [features["helix_af_hom"], features["helix_af_het"]],
    axis=1
).max(axis=1)

features["helix_mean_arf"] = to_num(master["helix_mean_arf"], default=0).clip(lower=0)
features["helix_max_arf"] = to_num(master["helix_max_arf"], default=0).clip(lower=0)

features["log_helix_counts_hom"] = log10_count(features["helix_counts_hom"])
features["log_helix_counts_het"] = log10_count(features["helix_counts_het"])
features["log_helix_total_counts"] = log10_count(features["helix_total_counts"])
features["log_helix_af_hom"] = log10_af(features["helix_af_hom"])
features["log_helix_af_het"] = log10_af(features["helix_af_het"])
features["log_helix_combined_af"] = log10_af(features["helix_combined_af"])

features["helix_has_homoplasmic_carrier"] = (features["helix_counts_hom"] > 0).astype(int)
features["helix_has_heteroplasmic_carrier"] = (features["helix_counts_het"] > 0).astype(int)

features[[
    "helix_observed", "helix_total_counts", "helix_combined_af",
    "helix_mean_arf", "helix_max_arf"
]].describe().T

## 12. Haplogroup/population features

These are not included in the main neutral matrix by default.

In [ ]:
features["hap_defining_variant"] = to_bool01(master["hap_defining_variant"])

hom_haps = master["helix_haplogroups_hom"].astype("string").fillna("").str.strip()
het_haps = master["helix_haplogroups_het"].astype("string").fillna("").str.strip()

features["has_helix_haplogroups_hom"] = (~hom_haps.eq("") & ~hom_haps.str.lower().eq("nan")).astype(int)
features["has_helix_haplogroups_het"] = (~het_haps.eq("") & ~het_haps.str.lower().eq("nan")).astype(int)

features[["hap_defining_variant", "has_helix_haplogroups_hom", "has_helix_haplogroups_het"]].sum()

## 13. Consequence and broad region features

In [ ]:
consequence = master["consequence"].astype("string").fillna("").str.lower()
vep = master["vep"].astype("string").fillna("").str.lower()
feature = master["feature"].astype("string").fillna("").str.lower()
gene = master["gene"].astype("string").fillna("").str.lower()

combined = consequence + " " + vep + " " + feature + " " + gene

features["is_intergenic"] = combined.str.contains("intergenic", regex=False).astype(int)
features["is_non_coding"] = combined.str.contains(
    "non_coding|non-coding|non coding|ncrna|regulatory",
    regex=True
).astype(int)
features["is_synonymous"] = combined.str.contains("synonymous", regex=False).astype(int)
features["is_missense"] = combined.str.contains("missense", regex=False).astype(int)
features["is_stop_gained"] = combined.str.contains("stop_gained|stop gained|stop-gained", regex=True).astype(int)
features["is_start_lost"] = combined.str.contains("start_lost|start lost|start-lost", regex=True).astype(int)
features["is_frameshift"] = combined.str.contains("frameshift", regex=False).astype(int)
features["is_splice"] = combined.str.contains("splice", regex=False).astype(int)

features["is_tRNA"] = (
    combined.str.contains("trna", regex=False)
    | gene.str.upper().str.startswith("MT-T", na=False)
).astype(int)

features["is_rRNA"] = (
    combined.str.contains("rrna", regex=False)
    | gene.str.upper().isin(["MT-RNR1", "MT-RNR2", "RNR1", "RNR2"])
).astype(int)

features["is_D_loop"] = combined.str.contains(
    "d-loop|dloop|control region|control_region|hvr|mt-cr|mt-crb",
    regex=True
).astype(int)

features["is_coding"] = (
    features["is_synonymous"].eq(1)
    | features["is_missense"].eq(1)
    | features["is_stop_gained"].eq(1)
    | features["is_start_lost"].eq(1)
    | combined.str.contains("protein_coding|coding|cds|mt-co|mt-cox|mt-cyb|mt-nd|mt-atp", regex=True)
).astype(int)

known_region = features[[
    "is_coding", "is_tRNA", "is_rRNA", "is_D_loop",
    "is_intergenic", "is_non_coding"
]].sum(axis=1) > 0

features["is_other_region"] = (~known_region).astype(int)

features[[
    "is_intergenic", "is_non_coding", "is_coding",
    "is_tRNA", "is_rRNA", "is_D_loop", "is_other_region"
]].sum()

## 14. Constraint block features

For constraint numeric columns, we median-impute missing values and create missingness indicators.

In [ ]:
constraint_prefixes = ["gene_constraint", "regional_constraint", "noncoding_constraint"]
constraint_suffixes = ["observed", "expected", "obs_exp", "lower_ci", "upper_ci"]

for prefix in constraint_prefixes:
    for suffix in constraint_suffixes:
        col = f"{prefix}_{suffix}"
        x = to_num(master[col]) if col in master.columns else pd.Series(np.nan, index=master.index)
        features[col] = x
        features[f"{col}_missing"] = x.isna().astype(int)
        med = x.median(skipna=True)
        if pd.isna(med):
            med = 0
        features[col] = x.fillna(med)

features["in_regional_constraint"] = to_bool01(master["in_regional_constraint"])
features["in_noncoding_constraint"] = to_bool01(master["in_noncoding_constraint"])

features["has_gene_constraint"] = ~master["gene_constraint_symbol"].astype("string").fillna("").str.strip().eq("")
features["has_regional_constraint"] = ~master["regional_constraint_symbol"].astype("string").fillna("").str.strip().eq("")
features["has_noncoding_constraint"] = ~master["noncoding_constraint_locus"].astype("string").fillna("").str.strip().eq("")

features["has_gene_constraint"] = features["has_gene_constraint"].astype(int)
features["has_regional_constraint"] = features["has_regional_constraint"].astype(int)
features["has_noncoding_constraint"] = features["has_noncoding_constraint"].astype(int)

features[[
    "gene_constraint_obs_exp", "regional_constraint_obs_exp", "noncoding_constraint_obs_exp",
    "in_regional_constraint", "in_noncoding_constraint",
    "has_gene_constraint", "has_regional_constraint", "has_noncoding_constraint"
]].head()

## 15. Labels and artifact flags for interpretation only

These variables should not be used in the main neutral unsupervised matrix.

In [ ]:
features["is_disease_suspected_dataset3"] = to_bool01(master["is_disease_suspected_dataset3"])
features["is_neutral_dataset8"] = to_bool01(master["is_neutral_dataset8"])
features["is_pathogenic_dataset9"] = to_bool01(master["is_pathogenic_dataset9"])
features["is_artifact_prone_site"] = to_bool01(master["is_artifact_prone_site"])

validation = master["validation_label"].astype("string").fillna("").str.strip().str.lower()

features["validation_label_is_unlabeled"] = validation.eq("unlabeled").astype(int)
features["validation_label_is_neutral"] = validation.str.contains("neutral", regex=False).astype(int)
features["validation_label_is_pathogenic"] = validation.str.contains("pathogenic", regex=False).astype(int)
features["validation_label_is_disease_suspected"] = validation.str.contains("disease|suspected", regex=True).astype(int)

features[[
    "is_disease_suspected_dataset3", "is_neutral_dataset8",
    "is_pathogenic_dataset9", "is_artifact_prone_site",
    "validation_label_is_unlabeled", "validation_label_is_neutral",
    "validation_label_is_pathogenic", "validation_label_is_disease_suspected"
]].sum()

## 16. Build modeling matrices

In [ ]:
neutral_columns = [
    "variant_id",

    # MLC / site constraint
    "mlc_score",
    "mlc_position_score",
    "mlc_score_missing",
    "mlc_position_score_missing",

    # gnomAD
    "gnomad_observed",
    "gnomad_filter_pass",
    "gnomad_filter_missing",
    "log_gnomad_homoplasmic_af",
    "log_gnomad_heteroplasmic_af",
    "log_gnomad_combined_af",
    "log_gnomad_total_ac",
    "log_gnomad_AN",
    "gnomad_max_heteroplasmy",
    "gnomad_has_homoplasmic_carrier",
    "gnomad_has_heteroplasmic_carrier",

    # Helix
    "helix_observed",
    "log_helix_counts_hom",
    "log_helix_counts_het",
    "log_helix_total_counts",
    "log_helix_af_hom",
    "log_helix_af_het",
    "log_helix_combined_af",
    "helix_mean_arf",
    "helix_max_arf",
    "helix_has_homoplasmic_carrier",
    "helix_has_heteroplasmic_carrier",

    # Variant type
    "is_snv",
    "is_indel",
    "is_transition",
    "is_transversion",

    # Consequence / broad region
    "is_intergenic",
    "is_non_coding",
    "is_synonymous",
    "is_missense",
    "is_stop_gained",
    "is_start_lost",
    "is_frameshift",
    "is_splice",
    "is_coding",
    "is_tRNA",
    "is_rRNA",
    "is_D_loop",
    "is_other_region",

    # Constraint blocks
    "gene_constraint_observed",
    "gene_constraint_expected",
    "gene_constraint_obs_exp",
    "gene_constraint_lower_ci",
    "gene_constraint_upper_ci",
    "gene_constraint_observed_missing",
    "gene_constraint_expected_missing",
    "gene_constraint_obs_exp_missing",
    "gene_constraint_lower_ci_missing",
    "gene_constraint_upper_ci_missing",

    "regional_constraint_observed",
    "regional_constraint_expected",
    "regional_constraint_obs_exp",
    "regional_constraint_lower_ci",
    "regional_constraint_upper_ci",
    "regional_constraint_observed_missing",
    "regional_constraint_expected_missing",
    "regional_constraint_obs_exp_missing",
    "regional_constraint_lower_ci_missing",
    "regional_constraint_upper_ci_missing",

    "noncoding_constraint_observed",
    "noncoding_constraint_expected",
    "noncoding_constraint_obs_exp",
    "noncoding_constraint_lower_ci",
    "noncoding_constraint_upper_ci",
    "noncoding_constraint_observed_missing",
    "noncoding_constraint_expected_missing",
    "noncoding_constraint_obs_exp_missing",
    "noncoding_constraint_lower_ci_missing",
    "noncoding_constraint_upper_ci_missing",

    "in_regional_constraint",
    "in_noncoding_constraint",
    "has_gene_constraint",
    "has_regional_constraint",
    "has_noncoding_constraint",

    # Circular position
    "mt_pos_sin",
    "mt_pos_cos",
]

neutral_matrix = features[neutral_columns].copy()

neutral_no_position_matrix = neutral_matrix.drop(
    columns=["mt_pos_sin", "mt_pos_cos"]
).copy()

population_matrix = features[
    neutral_columns
    + [
        "hap_defining_variant",
        "has_helix_haplogroups_hom",
        "has_helix_haplogroups_het",
    ]
].copy()

enriched_matrix = features[
    neutral_columns
    + [
        "hap_defining_variant",
        "has_helix_haplogroups_hom",
        "has_helix_haplogroups_het",
        "is_disease_suspected_dataset3",
        "is_neutral_dataset8",
        "is_pathogenic_dataset9",
        "is_artifact_prone_site",
        "validation_label_is_unlabeled",
        "validation_label_is_neutral",
        "validation_label_is_pathogenic",
        "validation_label_is_disease_suspected",
    ]
].copy()

interpretation_table = features[[
    "variant_id", "position", "reference", "alternate", "consequence", "gene", "feature",
    "mlc_score", "mlc_position_score",
    "gnomad_observed", "gnomad_filter_pass", "gnomad_AN",
    "gnomad_homoplasmic_ac", "gnomad_heteroplasmic_ac", "gnomad_total_ac",
    "gnomad_homoplasmic_af", "gnomad_heteroplasmic_af", "gnomad_combined_af",
    "gnomad_max_heteroplasmy",
    "helix_observed", "helix_counts_hom", "helix_counts_het", "helix_total_counts",
    "helix_af_hom", "helix_af_het", "helix_combined_af", "helix_mean_arf", "helix_max_arf",
    "hap_defining_variant",
    "is_intergenic", "is_non_coding", "is_synonymous", "is_missense", "is_stop_gained",
    "is_coding", "is_tRNA", "is_rRNA", "is_D_loop",
    "gene_constraint_obs_exp", "regional_constraint_obs_exp", "noncoding_constraint_obs_exp",
    "in_regional_constraint", "in_noncoding_constraint",
    "is_disease_suspected_dataset3", "is_neutral_dataset8", "is_pathogenic_dataset9",
    "is_artifact_prone_site",
    "validation_label_is_unlabeled", "validation_label_is_neutral",
    "validation_label_is_pathogenic", "validation_label_is_disease_suspected",
]].copy()

neutral_matrix.shape, neutral_no_position_matrix.shape, population_matrix.shape, enriched_matrix.shape

## 17. Validate matrices

Before saving, check that modeling matrices contain only numeric columns after `variant_id`, with no `NaN`, no infinite values, and no duplicated `variant_id`.

In [ ]:
def validate_matrix(df, name):
    print(f"Checking {name}: {df.shape}")

    assert "variant_id" in df.columns, "variant_id is missing"
    assert df["variant_id"].duplicated().sum() == 0, "duplicated variant_id values"

    X = df.drop(columns=["variant_id"])

    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    assert len(non_numeric) == 0, f"non-numeric columns: {non_numeric}"

    n_missing = int(X.isna().sum().sum())
    assert n_missing == 0, f"missing values: {n_missing}"

    n_inf = int(np.isinf(X.to_numpy(dtype=float)).sum())
    assert n_inf == 0, f"infinite values: {n_inf}"

    print("OK")

validate_matrix(neutral_matrix, "neutral_matrix")
validate_matrix(neutral_no_position_matrix, "neutral_no_position_matrix")
validate_matrix(population_matrix, "population_matrix")
validate_matrix(enriched_matrix, "enriched_matrix")

## 18. Save outputs

In [ ]:
master.to_csv(OUT_DIR / "mtDNA_master_standardized.csv", index=False)
features.to_csv(OUT_DIR / "mtDNA_variant_features_full.csv", index=False)

neutral_matrix.to_csv(OUT_DIR / "mtDNA_variant_clustering_matrix_neutral.csv", index=False)
neutral_no_position_matrix.to_csv(OUT_DIR / "mtDNA_variant_clustering_matrix_neutral_no_position.csv", index=False)
population_matrix.to_csv(OUT_DIR / "mtDNA_variant_clustering_matrix_with_population.csv", index=False)
enriched_matrix.to_csv(OUT_DIR / "mtDNA_variant_clustering_matrix_enriched.csv", index=False)
interpretation_table.to_csv(OUT_DIR / "mtDNA_variant_interpretation_table.csv", index=False)

print("Saved to:", OUT_DIR)
for p in sorted(OUT_DIR.glob("mtDNA_variant*.csv")):
    print(p)

## 19. Simple QC report

In [ ]:
qc_lines = []

qc_lines.append("mtDNA variant feature matrix QC report")
qc_lines.append("=====================================")
qc_lines.append("")
qc_lines.append(f"Raw master rows: {master.shape[0]}")
qc_lines.append(f"Full feature table: {features.shape}")
qc_lines.append(f"Neutral matrix: {neutral_matrix.shape}")
qc_lines.append(f"Neutral no-position matrix: {neutral_no_position_matrix.shape}")
qc_lines.append(f"Population matrix: {population_matrix.shape}")
qc_lines.append(f"Enriched matrix: {enriched_matrix.shape}")
qc_lines.append("")

qc_lines.append("Binary feature counts")
qc_lines.append("---------------------")

count_cols = [
    "gnomad_observed",
    "gnomad_filter_pass",
    "gnomad_has_homoplasmic_carrier",
    "gnomad_has_heteroplasmic_carrier",
    "helix_observed",
    "helix_has_homoplasmic_carrier",
    "helix_has_heteroplasmic_carrier",
    "hap_defining_variant",
    "is_snv",
    "is_indel",
    "is_transition",
    "is_transversion",
    "is_intergenic",
    "is_non_coding",
    "is_synonymous",
    "is_missense",
    "is_coding",
    "is_tRNA",
    "is_rRNA",
    "is_D_loop",
    "in_regional_constraint",
    "in_noncoding_constraint",
    "is_disease_suspected_dataset3",
    "is_neutral_dataset8",
    "is_pathogenic_dataset9",
    "is_artifact_prone_site",
]

for col in count_cols:
    if col in features.columns:
        qc_lines.append(f"{col}: {int(features[col].sum())}")

qc_lines.append("")
qc_lines.append("Missing values in full feature table")
qc_lines.append("------------------------------------")

missing_features = features.isna().sum()
missing_features = missing_features[missing_features > 0].sort_values(ascending=False)

if len(missing_features) == 0:
    qc_lines.append("No missing values.")
else:
    for col, n in missing_features.items():
        qc_lines.append(f"{col}: {int(n)}")

qc_path = OUT_DIR / "mtDNA_variant_feature_qc_report.txt"
qc_path.write_text("\n".join(qc_lines), encoding="utf-8")

print(qc_path)
print("\n".join(qc_lines[:40]))

## 20. Quick look at final primary matrix

In [ ]:
neutral_matrix.head()

In [ ]:
neutral_matrix.drop(columns=["variant_id"]).describe().T